# Prepare Human Evaluation Files

Generates pre-filled CSV files for human annotators by:
1. Sampling questions from both datasets
2. Generating chatbot answers
3. Exporting CSVs with blank score columns ready for annotation

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import pandas as pd
from dotenv import load_dotenv
load_dotenv()

OUT_DIR = "evaluation_details/human_eval"

## GRADE Sample

In [ ]:
grade_df = pd.read_csv("evaluation_details/evaluation_gradebook/full_dataset_gradebook.csv")

# Stratified sample: ~10 per difficulty level
grade_sample = (
    grade_df.groupby("PerceivedExpertise", group_keys=False)
    .apply(lambda g: g.sample(min(10, len(g)), random_state=42))
    .reset_index(drop=True)
)
print(f"GRADE sample: {len(grade_sample)} rows")
print(grade_sample["PerceivedExpertise"].value_counts())

In [ ]:
# Generate chatbot answers
from grade_rag import ask_grade, get_grade_contexts  # requires grade_rag to be run first

grade_sample["chatbot_answer"] = grade_sample["Question"].apply(ask_grade)
grade_sample["retrieved_contexts"] = grade_sample["Question"].apply(
    lambda q: " ||| ".join(get_grade_contexts(q, k=3))
)

# Add blank annotation columns
for col in ["accuracy_1_5", "completeness_1_5", "clarity_1_5", "faithfulness_1_5", "notes"]:
    grade_sample[col] = ""

out_cols = ["Reference", "Topic", "PerceivedExpertise", "Question", "Answer",
            "chatbot_answer", "retrieved_contexts",
            "accuracy_1_5", "completeness_1_5", "clarity_1_5", "faithfulness_1_5", "notes"]

grade_sample[out_cols].to_csv(f"{OUT_DIR}/human_eval_grade_sample.csv", index=False)
print(f"Saved: {OUT_DIR}/human_eval_grade_sample.csv")

## RoB2 Sample

In [ ]:
rob2_path = "evaluation_details/evaluation_rob2/full_dataset_rob2.csv"
if not os.path.exists(rob2_path):
    print("RoB2 dataset not found — run generate_rob2_dataset.ipynb first.")
else:
    rob2_df = pd.read_csv(rob2_path)

    rob2_sample = (
        rob2_df.groupby("difficulty", group_keys=False)
        .apply(lambda g: g.sample(min(10, len(g)), random_state=42))
        .reset_index(drop=True)
    )
    print(f"RoB2 sample: {len(rob2_sample)} rows")
    print(rob2_sample["difficulty"].value_counts())

    from rob2_rag import ask_rob2, get_rob2_contexts  # requires rob2_rag to be run first

    rob2_sample["chatbot_answer"] = rob2_sample["question"].apply(ask_rob2)
    rob2_sample["retrieved_contexts"] = rob2_sample["question"].apply(
        lambda q: " ||| ".join(get_rob2_contexts(q, k=3))
    )

    for col in ["accuracy_1_5", "completeness_1_5", "clarity_1_5", "faithfulness_1_5", "notes"]:
        rob2_sample[col] = ""

    out_cols = ["id", "domain", "difficulty", "question", "reference_answer",
                "chatbot_answer", "retrieved_contexts",
                "accuracy_1_5", "completeness_1_5", "clarity_1_5", "faithfulness_1_5", "notes"]

    rob2_sample[out_cols].to_csv(f"{OUT_DIR}/human_eval_rob2_sample.csv", index=False)
    print(f"Saved: {OUT_DIR}/human_eval_rob2_sample.csv")